[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb)

# 🟢 Easy: Gradient Accumulation

Implement a **training step with gradient accumulation** — simulating large batches with limited memory.

### Signature
```python
def accumulated_step(model, optimizer, loss_fn, micro_batches) -> float:
    # micro_batches: list of (input, target) tuples
    # Returns: average loss (float)
```

### Algorithm
1. `optimizer.zero_grad()`
2. For each `(x, y)` in micro_batches: `loss = loss_fn(model(x), y) / len(micro_batches)`, then `loss.backward()`
3. `optimizer.step()`
4. Return total accumulated loss

The key insight: dividing each loss by `n` before backward makes accumulated gradients equal to a single large-batch gradient.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

## Gradient Accumulation
GPU 메모리가 부족할 때 쓰는 기법임.
원래는 batch를 크게 해서 한번에 학습하면 좋은데 이렇게 하면 메모리가 터질 수 있음.

ex. batch 128으로 1번 학습
= batch 32로 4번 학습

작은 크기로 나눠서 하면 gradient가 4분의 1씩만 계산됨 -> gradient를 버리지 않고, 축적(accumulate)해서 128짜리를 한번 한것처럼 만들자.


Algorithm

backward를 여러번 호출하면 gradient가 자동으로 누적됨!!

1. `optimizer.zero_grad()`

2. For each `(x, y)` in micro_batches: `loss = loss_fn(model(x), y) / len(micro_batches)`, then `loss.backward()`

backward를 여러번 호출하면 gradient가 자동으로 누적됨!! zero_grad()를 안하면 덮어쓰는게 아니라 더해짐. 이걸 의도적으로 이용하는 것임

3. `optimizer.step()`

4. Return total accumulated loss

The key insight: dividing each loss by n before backward makes accumulated gradients equal to a single large-batch gradient.

왜 이렇게하냐?
ex. 큰 배치 128개를 한번에 하면 loss = (다 더한 값 / 128

32개씩 4번 나눠서 하면 loss1 = () / 32, loss2 = () / 32, ... 이렇게 각 배치 내에서 평균됨.

이걸 그냥 누적해버리면 gradient = loss1 + loss2 + loss3 + loss4 -> 4배로 커져버림

그래서 n으로 나눠서 128개를 한번에 한 것과 동일하게 만들어줘야 함

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

# micro_batches = 작은 배치들의 리스트 [(x1, y1), (x2, y2), ...] 형태
def accumulated_step(model, optimizer, loss_fn, micro_batches):
    # zero_grad, loop (forward, scale loss, backward), step

    # 시작 전에 gradient를 initialize
    # 이걸 루프 밖에서 한 번만 해줘야 함!
    # 루프 안에서 하면 매번 gradient가 리셋돼서 누적이 안 됨
    optimizer.zero_grad()

    total_loss = 0.0
    n = len(micro_batches)  # 나중에 총 개수로 나눠줄거니까 미리 저장해두기

    # micro_batch 하나씩 꺼내와서
    for x, y in micro_batches:
      # model(x) 은 forward (순전파)
      # loss_fn은 loss 계산 (모델 출력값과 정답값을 넣어줌)
      # /n 해주기 (나중에 합산 시 큰 배치랑 같아지도록 스케일 맞추는 작업)
      loss = loss_fn(model(x), y) / n

      # 여기서 zero_grad() 안하는게 핵심 -> gradient 누적
      loss.backward()

      # .item()으로 파이썬 숫자로 변환해서 더함
      total_loss += loss.item()

    # 루프 밖에서, accumulate된 gradient로 가중치를 한번만 업데이트 해줌
    optimizer.step()

    return total_loss

In [4]:
# 🧪 Debug
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss = accumulated_step(model, opt, nn.MSELoss(),
    [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)])
print('Loss:', loss)

Loss: 0.6689327545464039


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_accumulation')


🧪 Testing: Gradient Accumulation (Easy)
──────────────────────────────────────────────────
  ✅ [1/3] Matches full batch update (35.5ms)
  ✅ [2/3] Returns loss value (1.1ms)
  ✅ [3/3] Parameters actually update (0.7ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (37.4ms total)
  Progress saved. Run status() to see your dashboard.

